In [1248]:
import pandas as pd
import json

In [1249]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# หาหน่วยเลือกตั้งของจังหวัด
rayong = next((p for p in data['provinces'] if p['name'] == 'ระยอง'), None)
# print(f"ระยอง {rayong['total_stations']} หน่วยเลือกตั้ง")
d = dict()
sd2d = dict()
t = 0
# หาหน่วยเลือกตั้งจากรหัส
def find_total_station(area,province):
    for p in data['provinces']:
        if p['name'] == province:
            for a in p['areas']:
                if a['area'] == area:
                    for unit in a['stations']:
                        if unit['district'] not in d:
                            d[unit['district']] = dict()
                        if unit['subdistrict'] not in  d[unit['district']]:
                            d[unit['district']][unit['subdistrict']] = 1
                            sd2d[unit['subdistrict']] = unit['district']
                        else:
                            d[unit['district']][unit['subdistrict']] += 1
    return None

find_total_station(2, 'ลำปาง')
print(d)
print(sd2d)
for k, v in d.items():
    for e, g in v.items():
        t+=g
print(t)

{'เมืองลำปาง': {'บ้านแลง': 12, 'บ้านเสด็จ': 19}, 'งาว': {'หลวงเหนือ': 7, 'หลวงใต้': 8, 'บ้านโป่ง': 12, 'บ้านร้อง': 13, 'ปงเตา': 13, 'นาแก': 6, 'บ้านอ้อน': 8, 'บ้านแหง': 9, 'บ้านหวด': 7, 'แม่ตีบ': 7}, 'แจ้ห่ม': {'แจ้ห่ม': 14, 'บ้านสา': 10, 'ปงดอน': 9, 'แม่สุก': 12, 'เมืองมาย': 7, 'ทุ่งผึ้ง': 8, 'วิเชตนคร': 12}, 'วังเหนือ': {'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}, 'เมืองปาน': {'เมืองปาน': 11, 'บ้านขอ': 14, 'ทุ่งกว๋าว': 16, 'แจ้ซ้อน': 15, 'หัวเมือง': 9}}
{'บ้านแลง': 'เมืองลำปาง', 'บ้านเสด็จ': 'เมืองลำปาง', 'หลวงเหนือ': 'งาว', 'หลวงใต้': 'งาว', 'บ้านโป่ง': 'งาว', 'บ้านร้อง': 'งาว', 'ปงเตา': 'งาว', 'นาแก': 'งาว', 'บ้านอ้อน': 'งาว', 'บ้านแหง': 'งาว', 'บ้านหวด': 'งาว', 'แม่ตีบ': 'งาว', 'แจ้ห่ม': 'แจ้ห่ม', 'บ้านสา': 'แจ้ห่ม', 'ปงดอน': 'แจ้ห่ม', 'แม่สุก': 'แจ้ห่ม', 'เมืองมาย': 'แจ้ห่ม', 'ทุ่งผึ้ง': 'แจ้ห่ม', 'วิเชตนคร': 'แจ้ห่ม', 'ทุ่งฮั้ว': 'วังเหนือ', 'วังเหนือ': 'วังเหนือ', 'วังใต้': 'วังเหนือ', 'ร่องเคาะ': 'วังเ

In [1250]:
d['นอกเขต'] = dict()
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    sd2d[sub_dist] = 'นอกเขต'
    d['นอกเขต'][sub_dist] = 1
    # all_subdistrict.append(sub_dist)

In [1251]:
d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
d['แจ้ห่ม']['แจ้ห่ม(ในเขต)'] = 6
d['แจ้ห่ม']['แจ้ห่ม(นอกเขต)'] = 8
d['แจ้ห่ม'].pop('แจ้ห่ม', None)
sd2d['บ้านใหม่'] = 'วังเหนือ'
sd2d['แจ้ห่ม(ในเขต)'] = "แจ้ห่ม"
sd2d['แจ้ห่ม(นอกเขต)'] = "แจ้ห่ม"

In [1252]:
all_distrcit = []
all_subdistrict = []
district_count = {}
for k,v in d.items():
    all_distrcit.append(k)
    all_subdistrict += v.keys()
    for key in v.keys():
        if k not in district_count:
            district_count[k] = v[key]
        else: district_count[k]+= v[key]

In [1253]:
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    all_subdistrict.append(sub_dist)

In [1254]:
df = pd.read_csv("../ocr-result1/master_results.csv")

In [1255]:
wrong_districts = []
for wrong_district in df['district'].unique().tolist():
    if wrong_district not in all_distrcit and wrong_district != 'นอกเขต':
        wrong_districts.append(wrong_district)

wrong_districts

[]

In [1256]:
from rapidfuzz import process
def automate_edit(feature, check, wrongs, df):
    wrong2correct = dict()
    scores = dict()
    lis = []
    for text in wrongs:
        match = process.extractOne(text, check, score_cutoff=50)

        if match:
            result, score, index = match
            wrong2correct[text] = result
            scores[text] = round(score,2)
            if scores[text] >= 70:
                df.loc[df[feature] == text, feature] = result
            else:
                lis.append(text)
            print(f"OCR: '{text}' -> Corrected: '{result}' (Confidence: {score:.2f}%)")
        else:
            wrong2correct[text] = "can't find"
            scores[text] = 0
            lis.append(text)
            print(f"OCR: '{text}' -> [Manual Review Needed]")
    return wrong2correct, scores, lis

In [1257]:
df_test = df.copy()
wrong2correct, scores, list_wrongs = automate_edit('district', all_distrcit, wrong_districts, df_test)

In [1258]:
wrong_sds = []
for wrong_sd in df_test['sub-district'].unique().tolist():
    if wrong_sd not in all_subdistrict:
        wrong_sds.append(wrong_sd)

wrong_sds

[]

In [1259]:
df_test.loc[df_test['sub-district'] == 'เเจ้ซ้อน', 'sub-district'] = 'แจ้ซ้อน'

In [1260]:
l = df_test['sub-district'].unique().tolist()
for sd in all_subdistrict:
    if sd not in l:
        print(f"เขต {sd} หายไปทั้ง บช และ เขต")

In [1261]:
df = df_test.copy()

In [1262]:
df['district'].unique().tolist()

['นอกเขต', 'งาว', 'วังเหนือ', 'เมืองปาน', 'เมืองลำปาง', 'แจ้ห่ม']

In [1263]:
correct_names = [
    "นางสาววิชุดา ว่องวัฒนวิโรจน์",
    "นางสาวสุวิภา กุศลจูง",
    "นายศรีพรหม หอมยก",
    "นายสมบูรณ์ รูปสะอาด",
    "นายดาชัย เอกปฐพี",
    "นายธนาธร โล่ห์สุนทร",
    "พันเอกสันทัด ภัทรกิตตินนท์",
]

In [1264]:
party2number = dict()
with open("../partys_lists_number.txt", 'r', encoding="utf-8") as f:
    lines = []
    for line in f:
        if line.strip() == "":continue
        num, party = line.strip().split(":")
        num = int(num.strip().split()[1])
        party = party.strip().removeprefix("พรรค")
        lines.append(party)
        party2number[party] = num
for i in range(1,8):
    party2number[correct_names[i-1]] = i
correct_names += lines

In [1265]:
correct_names

['นางสาววิชุดา ว่องวัฒนวิโรจน์',
 'นางสาวสุวิภา กุศลจูง',
 'นายศรีพรหม หอมยก',
 'นายสมบูรณ์ รูปสะอาด',
 'นายดาชัย เอกปฐพี',
 'นายธนาธร โล่ห์สุนทร',
 'พันเอกสันทัด ภัทรกิตตินนท์',
 'ไทยทรัพย์ทวี',
 'เพื่อชาติไทย',
 'ใหม่',
 'มิติใหม่',
 'รวมใจไทย',
 'รวมไทยสร้างชาติ',
 'พลวัต',
 'ประชาธิปไตยใหม่',
 'เพื่อไทย',
 'ทางเลือกใหม่',
 'เศรษฐกิจ',
 'เสรีรวมไทย',
 'รวมพลังประชาชน',
 'ท้องที่ไทย',
 'อนาคตไทย',
 'พลังเพื่อไทย',
 'ไทยชนะ',
 'พลังสังคมใหม่',
 'สังคมประชาธิปไตย',
 'ฟิวชัน',
 'ไทรวมพลัง',
 'ก้าวอิสระ',
 'ปวงชนไทย',
 'วิชชั่นใหม่',
 'เพื่อชีวิตใหม่',
 'คลองไทย',
 'ประชาธิปัตย์',
 'ไทยก้าวหน้า',
 'ไทยภักดี',
 'แรงงานสร้างชาติ',
 'ประชากรไทย',
 'ครูไทยเพื่อประชาชน',
 'ประชาชาติ',
 'สร้างอนาคตไทย',
 'รักชาติ',
 'ไทยพร้อม',
 'ภูมิใจไทย',
 'พลังธรรมใหม่',
 'กรีน',
 'ไทยธรรม',
 'แผ่นดินธรรม',
 'กล้าธรรม',
 'พลังประชารัฐ',
 'โอกาสใหม่',
 'เป็นธรรม',
 'ประชาชน',
 'ประชาไทย',
 'ไทยสร้างไทย',
 'ไทยก้าวใหม่',
 'ประชาอาสาชาติ',
 'พร้อม',
 'เครือข่ายชาวนาแห่งประเทศไทย',
 'ไทยพิทักษ์ธรรม',
 'ความหวั

In [1266]:
df_test = df.copy()

In [1267]:
wrong_names = []
for wrong_name in df_test['name'].unique().tolist():
    if wrong_name not in all_subdistrict:
        wrong_names.append(wrong_name)

wrong_names

['ไทยทรัพย์ทวี',
 'เพื่อชาติไทย',
 'ใหม่',
 'มิติใหม่',
 'รวมใจไทย',
 'รวมไทยสร้างชาติ',
 'พลวัต',
 'ประชาธิปไตยใหม่',
 'เพื่อไทย',
 'ทางเลือกใหม่',
 'เศรษฐกิจ',
 'เสรีรวมไทย',
 'รวมพลังประชาชน',
 'ท้องที่ไทย',
 'อนาคตไทย',
 'พลังเพื่อไทย',
 'ไทยชนะ',
 'พลังสังคมใหม่',
 'สังคมประชาธิปไตย',
 'ฟิวชัน',
 'ไทรวมพลัง',
 'ก้าวอิสระ',
 'ปวงชนไทย',
 'วิชชั่นใหม่',
 'เพื่อชีวิตใหม่',
 'คลองไทย',
 'ประชาธิปัตย์',
 'ไทยก้าวหน้า',
 'ไทยภักดี',
 'แรงงานสร้างชาติ',
 'ประชากรไทย',
 'ครูไทยเพื่อประชาชน',
 'ประชาชาติ',
 'สร้างอนาคตไทย',
 'รักชาติ',
 'ไทยพร้อม',
 'ภูมิใจไทย',
 'พลังธรรมใหม่',
 'กรีน',
 'ไทยธรรม',
 'แผ่นดินธรรม',
 'กล้าธรรม',
 'พลังประชารัฐ',
 'โอกาสใหม่',
 'เป็นธรรม',
 'ประชาชน',
 'ประชาไทย',
 'ไทยสร้างไทย',
 'ไทยก้าวใหม่',
 'ประชาอาสาชาติ',
 'พร้อม',
 'เครือข่ายชาวนาแห่งประเทศไทย',
 'ไทยพิทักษ์ธรรม',
 'ความหวังใหม่',
 'ไทยรวมไทย',
 'เพื่อบ้านเมือง',
 'พลังไทยรักชาติ',
 'นางสาววิชุดา ว่องวัฒนวิโรจน์',
 'นางสาวสุวิภา กุศลจูง',
 'นายศรีพรหม หอมยก',
 'นายสมบูรณ์ รูปสะอาด',
 'นายดาชัย เอกปฐ

In [1268]:
wrong2correct, scores, list_wrongs = automate_edit('name', correct_names, wrong_names, df_test)

OCR: 'ไทยทรัพย์ทวี' -> Corrected: 'ไทยทรัพย์ทวี' (Confidence: 100.00%)
OCR: 'เพื่อชาติไทย' -> Corrected: 'เพื่อชาติไทย' (Confidence: 100.00%)
OCR: 'ใหม่' -> Corrected: 'ใหม่' (Confidence: 100.00%)
OCR: 'มิติใหม่' -> Corrected: 'มิติใหม่' (Confidence: 100.00%)
OCR: 'รวมใจไทย' -> Corrected: 'รวมใจไทย' (Confidence: 100.00%)
OCR: 'รวมไทยสร้างชาติ' -> Corrected: 'รวมไทยสร้างชาติ' (Confidence: 100.00%)
OCR: 'พลวัต' -> Corrected: 'พลวัต' (Confidence: 100.00%)
OCR: 'ประชาธิปไตยใหม่' -> Corrected: 'ประชาธิปไตยใหม่' (Confidence: 100.00%)
OCR: 'เพื่อไทย' -> Corrected: 'เพื่อไทย' (Confidence: 100.00%)
OCR: 'ทางเลือกใหม่' -> Corrected: 'ทางเลือกใหม่' (Confidence: 100.00%)
OCR: 'เศรษฐกิจ' -> Corrected: 'เศรษฐกิจ' (Confidence: 100.00%)
OCR: 'เสรีรวมไทย' -> Corrected: 'เสรีรวมไทย' (Confidence: 100.00%)
OCR: 'รวมพลังประชาชน' -> Corrected: 'รวมพลังประชาชน' (Confidence: 100.00%)
OCR: 'ท้องที่ไทย' -> Corrected: 'ท้องที่ไทย' (Confidence: 100.00%)
OCR: 'อนาคตไทย' -> Corrected: 'อนาคตไทย' (Confidence: 100.00

In [1269]:
list_wrongs

[]

In [1270]:
wrong2correct = {
    "พิวซัน": 'ฟิวชัน',
    "ก้าวล่วง": 'ก้าวอิสระ',
    "พิวจัน":"ฟิวชัน",
    "พิกุลัน":"ฟิวชัน",
    "พิวจันทร์":"ฟิวชัน",
    "พิวซิน":"ฟิวชัน",
    "พิวขัน":"ฟิวชัน",
    "กษิน":"กรีน",
    "วิชัยขับใหญ่":"วิชชั่นใหม่",
    "พิชัย":"ฟิวชัน",
    "ทรอม":"พร้อม",
    "พืชขัน":"ฟิวชัน",
    "พืชชัน":"ฟิวชัน",
    "วิภาวดี":"รักชาติ",
    "กรุงเทพมหานคร":"กรีน",
    "ไทยทรัพยากร":"ไทยทรัพย์ทวี",
    "พลังประจวบ":"พลังประชารัฐ",
    "พิวัฒน์":"ฟิวชัน",
    "กวิน":"กรีน",
    "ดร.ณฐภัทร":"เศรษฐกิจ",
    "ศรีราชาไทย":"เสรีรวมไทย",
    "พิมพ์ขึ้น":"ฟิวชัน", 
    "โทรสารเคล็ง":"ไทรวมพลัง",
    "กำกวติสร":"ก้าวอิสระ",
    "พิกุลชน":"ฟิวชัน",
    "นางสาววิฤดา วงศ์พรมไพรณ์":correct_names[0],
    "พิกวัน":"ฟิวชัน",
    "พืชซัน":"ฟิวชัน",
    "พืชชน":"ฟิวชัน",
    "ฟิลิปปิน":"ฟิวชัน",
    "ไทกรุงเทพฯ":"ไทรวมพลัง",
    "ก้าวก้มลง":"ก้าวอิสระ",
    "พิกขัน":"ฟิวชัน",
    "พิกุล":"ฟิวชัน",
    "กำกวติสระ":"ก้าวอิสระ"
}

In [1271]:
for wrong in list_wrongs:
    print(df_test[df_test['name'] == wrong])

In [1272]:
for wrong, correct in wrong2correct.items():
    df_test.loc[df_test['name'] == wrong, 'name'] = correct

In [1273]:
for wrong in list_wrongs:
    print(df_test[df_test['name'] == wrong])

In [1274]:
for i in df_test['name'].unique().tolist():
    if i not in correct_names:
        print(i)

In [1275]:
df_test.shape

(21496, 7)

In [1276]:
df = df_test.copy()

In [1277]:
def check_from_df(df):
    expected_khet = set(correct_names[:7])
    expected_bch = set(correct_names[7:])
    
    df['name'] = df['name'].astype(str).str.strip()
    
    print("🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...\n")
    has_error = False
    
    # 3. จัดกลุ่มตรวจสอบ (เพิ่ม 'district' เข้าไปในวงเล็บ)
    grouped = df.groupby(['district', 'sub-district', 'unit_number', 'type'])
    total = 0
    
    # เพิ่ม district มารับค่าในลูปด้วย
    for (district, sub_district, unit_number, type_val), group_data in grouped:
        # ดึงชื่อออกมาเป็น List (เพื่อนับซ้ำ) และ Set (เพื่อหาตัวที่ขาด/เกิน)
        actual_names_list = group_data['name'].tolist()
        actual_names_set = set(actual_names_list)
        
        if type_val == 'เขต':
            expected_names = expected_khet
        elif type_val == 'บช':
            expected_names = expected_bch
        else:
            continue 
        
        expected_total = len(expected_names)
    
        # --- เริ่มกระบวนการตรวจสอบ 3 ชั้น ---
        
        # 1. เช็คขาด (Missing)
        missing = expected_names - actual_names_set
        
        # 2. เช็คแปลกปลอม (Extra/Unrecognized)
        extra = actual_names_set - expected_names
        
        # 3. เช็คซ้ำซ้อน (Duplicates)
        name_counts = group_data['name'].value_counts()
        duplicates = name_counts[name_counts > 1].index.tolist()
    
        # ถ้าเจอความผิดปกติอย่างใดอย่างหนึ่ง ให้ปริ้นท์แจ้งเตือน
        if missing or extra or duplicates:
            has_error = True
            
            # 📌 แก้ไขบรรทัด Print ให้แสดง "อำเภอ" ด้วย
            print(f"📌 [{type_val}] อำเภอ: {district} | ตำบล: {sub_district} | หน่วยที่: {unit_number}")
            print(f"   📊 มีข้อมูลทั้งหมด {len(actual_names_list)} บรรทัด (จากที่ควรมี {expected_total} บรรทัด)")
            
            if missing:
                print(f"   ❌ ขาดหายไป {len(missing)} รายชื่อ: {', '.join(sorted(missing))}")
                total += len(missing)
            if extra:
                print(f"   🚨 แปลกปลอม/ผิดสเปก {len(extra)} รายชื่อ: {', '.join(sorted(extra))}")
                
            if duplicates:
                print(f"   🔄 รายชื่อซ้ำซ้อน {len(duplicates)} รายการ:")
                for dup in duplicates:
                    print(f"      - '{dup}' โผล่มา {name_counts[dup]} บรรทัด")
                    
            print("-" * 60)
            
    print(f"จำนวนที่ขาดรวมคิดเป็น: {total}")
    if not has_error:
        print("✅ ข้อมูลสมบูรณ์แบบ! ทุกหน่วยมีรายชื่อครบถ้วน ไม่มีขาด ไม่มีเกิน ไม่มีซ้ำครับ")

In [1278]:
def edit_df(previous_val, old_val, new_val, df):

    previous_name = df.groupby(['district', 'sub-district', 'unit_number', 'type'])['name'].shift(1)

    condition_fix = (df['name'] == old_val) & (previous_name == previous_val)

    rows_to_fix = condition_fix.sum()

    if rows_to_fix > 0:
        df.loc[condition_fix, 'name'] = new_val
        print(f"✅ แก้ไขพรรค {old_val} เป็น {new_val} (ที่อยู่ต่อจาก {previous_val}) สำเร็จจำนวน {rows_to_fix} แถว")
    else:
        print(f"✅ ไม่พบเคสที่พรรค {old_val} อยู่ต่อจาก {previous_val}")

In [1279]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [1280]:
edit_df("ปวงชนไทย","ใหม่", "วิชชั่นใหม่", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ปวงชนไทย


In [1281]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [1282]:
edit_df("ไทยสร้างไทย", 'ใหม่', "ไทยก้าวใหม่", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ไทยสร้างไทย


In [1283]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [1284]:
edit_df("พลวัต", "ใหม่", "ประชาธิปไตยใหม่", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก พลวัต


In [1285]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [1286]:
df.shape

(21496, 7)

In [1287]:
print(df[(df['sub-district'] == 'ทุ่งฮั้ว') & (df['unit_number'] == 7)].to_string())

     type province  district sub-district  unit_number                          name score
6366  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7  นางสาววิชุดา ว่องวัฒนวิโรจน์    21
6367  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7          นางสาวสุวิภา กุศลจูง    57
6368  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7              นายศรีพรหม หอมยก     3
6369  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7           นายสมบูรณ์ รูปสะอาด     3
6370  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7              นายดาชัย เอกปฐพี    12
6371  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7           นายธนาธร โล่ห์สุนทร    45
6372  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7    พันเอกสันทัด ภัทรกิตตินนท์     2
7321   บช    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7                  ไทยทรัพย์ทวี     3
7322   บช    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7                  เพื่อชาติไทย     8
7323   บช    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7                          ใหม่     0

In [1288]:
edit_df("ก้าวอิสระ", "ประชาไทย", "ปวงชนไทย", df)
edit_df("ปวงชนไทย", "ใหม่", "วิชชั่นใหม่", df)
check_from_df(df)

✅ ไม่พบเคสที่พรรค ประชาไทย อยู่ต่อจาก ก้าวอิสระ
✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ปวงชนไทย
🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรั

In [1289]:
print(df[(df['sub-district'] == 'บ้านสา') & (df['unit_number'] == 1)].to_string())

      type province district sub-district  unit_number                          name score
17081  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1  นางสาววิชุดา ว่องวัฒนวิโรจน์    13
17082  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1          นางสาวสุวิภา กุศลจูง    99
17083  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1              นายศรีพรหม หอมยก    11
17084  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1           นายสมบูรณ์ รูปสะอาด     8
17085  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1              นายดาชัย เอกปฐพี    10
17086  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1           นายธนาธร โล่ห์สุนทร    54
17087  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1    พันเอกสันทัด ภัทรกิตตินนท์    11
17529   บช    ลำปาง   แจ้ห่ม       บ้านสา            1                  ไทยทรัพย์ทวี     0
17530   บช    ลำปาง   แจ้ห่ม       บ้านสา            1                  เพื่อชาติไทย     9
17531   บช    ลำปาง   แจ้ห่ม       บ้านสา            1                          ใหม่     1

In [1290]:
edit_df("วิชชั่นใหม่", "เพื่อชาติไทย", "เพื่อชีวิตใหม่", df)

✅ ไม่พบเคสที่พรรค เพื่อชาติไทย อยู่ต่อจาก วิชชั่นใหม่


In [1291]:
print(df[(df['sub-district'] == 'วังทอง') & (df['unit_number'] == 9)].to_string())

      type province  district sub-district  unit_number                          name score
6730   เขต    ลำปาง  วังเหนือ       วังทอง            9  นางสาววิชุดา ว่องวัฒนวิโรจน์     7
6731   เขต    ลำปาง  วังเหนือ       วังทอง            9          นางสาวสุวิภา กุศลจูง    48
6732   เขต    ลำปาง  วังเหนือ       วังทอง            9              นายศรีพรหม หอมยก     3
6733   เขต    ลำปาง  วังเหนือ       วังทอง            9           นายสมบูรณ์ รูปสะอาด     0
6734   เขต    ลำปาง  วังเหนือ       วังทอง            9              นายดาชัย เอกปฐพี   142
6735   เขต    ลำปาง  วังเหนือ       วังทอง            9           นายธนาธร โล่ห์สุนทร    26
6736   เขต    ลำปาง  วังเหนือ       วังทอง            9    พันเอกสันทัด ภัทรกิตตินนท์     4
10172   บช    ลำปาง  วังเหนือ       วังทอง            9                  ไทยทรัพย์ทวี     ☑
10173   บช    ลำปาง  วังเหนือ       วังทอง            9                  เพื่อชาติไทย     ☐
10174   บช    ลำปาง  วังเหนือ       วังทอง            9                         

In [1292]:
edit_df("ไทยทรัพย์ทวี", "ใหม่", "เพื่อชาติไทย", df)
edit_df("ใหม่", "ใหม่", "มิติใหม่", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ไทยทรัพย์ทวี
✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ใหม่


In [1293]:
print(df[(df['sub-district'] == 'เมืองมาย') & (df['unit_number'] == 1)].to_string())

      type province district sub-district  unit_number                          name score
17235  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1  นางสาววิชุดา ว่องวัฒนวิโรจน์     1
17236  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1          นางสาวสุวิภา กุศลจูง    19
17237  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1              นายศรีพรหม หอมยก     1
17238  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1           นายสมบูรณ์ รูปสะอาด     1
17239  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1              นายดาชัย เอกปฐพี   109
17240  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1           นายธนาธร โล่ห์สุนทร    25
17241  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1    พันเอกสันทัด ภัทรกิตตินนท์     1
19266   บช    ลำปาง   แจ้ห่ม     เมืองมาย            1                      เศรษฐกิจ     5
19267   บช    ลำปาง   แจ้ห่ม     เมืองมาย            1                    เสรีรวมไทย     4
19268   บช    ลำปาง   แจ้ห่ม     เมืองมาย            1                รวมพลังประชาชน     6

In [1294]:
edit_df("คลองไทย", "ประชาชน", "ประชาธิปัตย์", df)

✅ ไม่พบเคสที่พรรค ประชาชน อยู่ต่อจาก คลองไทย


In [1295]:
edit_df("คลองไทย", "ประชาชาติ", "ประชาธิปัตย์", df)

✅ ไม่พบเคสที่พรรค ประชาชาติ อยู่ต่อจาก คลองไทย


In [1296]:
print(df[(df['sub-district'] == 'แจ้ห่ม(นอกเขต)') & (df['unit_number'] == 7)].to_string())

      type province district    sub-district  unit_number                          name score
17452  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7  นางสาววิชุดา ว่องวัฒนวิโรจน์     9
17453  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7          นางสาวสุวิภา กุศลจูง   103
17454  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7              นายศรีพรหม หอมยก     4
17455  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7           นายสมบูรณ์ รูปสะอาด     9
17456  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7              นายดาชัย เอกปฐพี    80
17457  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7           นายธนาธร โล่ห์สุนทร    45
17458  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7    พันเอกสันทัด ภัทรกิตตินนท์     4
19977   บช    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7                  ไทยทรัพย์ทวี     3
19978   บช    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7                  เพื่อชาติไทย     9
19979   บช    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7   

In [1297]:
edit_df("สังคมประชาธิปไตย", "ใหม่", "ฟิวชัน", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก สังคมประชาธิปไตย


In [1298]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [1299]:
total = 0
for district in d:
    for sd, val in d[district].items():
        total += val

In [1300]:
total

355

In [1301]:
df['sub-district'] = df['sub-district'].astype(str).str.strip()
df['unit_number'] = pd.to_numeric(df['unit_number'], errors='coerce')

print("🔍 ตามล่าหาข้อมูลที่หายไป 'ทั้งก้อน'...")

missing_whole_groups = 0

# 1. เอา Group keys ที่มีอยู่จริงใน df มาเก็บไว้ใน Set (เพื่อให้ค้นหาได้เร็วๆ)
# หน้าตาข้อมูลใน Set จะเป็นแบบ: ('นาแก', 1, 'เขต'), ('นาแก', 1, 'บช')
existing_groups = set(tuple(x) for x in df[['sub-district', 'unit_number', 'type']].dropna().drop_duplicates().values)

# 2. วนลูปตามโครงสร้างความจริงจาก Dictionary d
for district in d:
    for sd, total_units in d[district].items():
        for unit in range(1, total_units + 1): # วนลูปหน่วยที่ 1 ถึง N
            c_unit = unit
            if district == 'นอกเขต':
                c_unit = -1
            # print(sd,c_unit)
            # เช็คว่า 'เขต' ของหน่วยนี้มีใน df ไหม?
            if (sd, c_unit, 'เขต') not in existing_groups:
                missing_whole_groups += 7
                print(f"🚨 หายไปทั้งหน้า! [เขต] ตำบล: {sd} | หน่วยที่: {c_unit} (หายไป 7 รายชื่อ)")
                
            # เช็คว่า 'บช' ของหน่วยนี้มีใน df ไหม?
            if (sd, c_unit, 'บช') not in existing_groups:
                missing_whole_groups += 57
                print(f"🚨 หายไปทั้งหน้า! [บช]  ตำบล: {sd} | หน่วยที่: {c_unit} (หายไป 57 รายชื่อ)")

print("-" * 50)
print(f"✅ จำนวนรายชื่อที่หายไปเพราะหายทั้งก้อน (ไม่ได้เข้า groupby): {missing_whole_groups}")

🔍 ตามล่าหาข้อมูลที่หายไป 'ทั้งก้อน'...
🚨 หายไปทั้งหน้า! [เขต] ตำบล: บ้านเสด็จ | หน่วยที่: 19 (หายไป 7 รายชื่อ)
🚨 หายไปทั้งหน้า! [เขต] ตำบล: บ้านโป่ง | หน่วยที่: 9 (หายไป 7 รายชื่อ)
🚨 หายไปทั้งหน้า! [บช]  ตำบล: บ้านโป่ง | หน่วยที่: 9 (หายไป 57 รายชื่อ)
🚨 หายไปทั้งหน้า! [เขต] ตำบล: บ้านโป่ง | หน่วยที่: 11 (หายไป 7 รายชื่อ)
🚨 หายไปทั้งหน้า! [บช]  ตำบล: ปงเตา | หน่วยที่: 4 (หายไป 57 รายชื่อ)
🚨 หายไปทั้งหน้า! [เขต] ตำบล: ทุ่งฮั้ว | หน่วยที่: 10 (หายไป 7 รายชื่อ)
--------------------------------------------------
✅ จำนวนรายชื่อที่หายไปเพราะหายทั้งก้อน (ไม่ได้เข้า groupby): 142


In [1302]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [1303]:
# df.to_csv("master_result1.csv", index=False)

In [1304]:
df.isnull().sum()

type              0
province          0
district          0
sub-district      0
unit_number       0
name              0
score           274
dtype: int64

In [1305]:
df_result2 = pd.read_csv("master_result2.csv")
df_result2.head()

,type,province,district,sub-district,unit_number,name,score
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ไทยทรัพย์ทวี,0
1,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,เพื่อชาติไทย,6
2,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ใหม่,4
3,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,มิติใหม่,0
4,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,รวมใจไทย,1


In [1306]:
check_from_df(df_result2)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

จำนวนที่ขาดรวมคิดเป็น: 0
✅ ข้อมูลสมบูรณ์แบบ! ทุกหน่วยมีรายชื่อครบถ้วน ไม่มีขาด ไม่มีเกิน ไม่มีซ้ำครับ


In [1307]:
def check_missing_allrecord(df):
    df['sub-district'] = df['sub-district'].astype(str).str.strip()
    df['unit_number'] = pd.to_numeric(df['unit_number'], errors='coerce')
    
    print("🔍 ตามล่าหาข้อมูลที่หายไป 'ทั้งก้อน'...")
    
    missing_whole_groups = 0
    
    # 1. เอา Group keys ที่มีอยู่จริงใน df มาเก็บไว้ใน Set (เพื่อให้ค้นหาได้เร็วๆ)
    # หน้าตาข้อมูลใน Set จะเป็นแบบ: ('นาแก', 1, 'เขต'), ('นาแก', 1, 'บช')
    existing_groups = set(tuple(x) for x in df[['sub-district', 'unit_number', 'type']].dropna().drop_duplicates().values)
    
    # 2. วนลูปตามโครงสร้างความจริงจาก Dictionary d
    for district in d:
        for sd, total_units in d[district].items():
            for unit in range(1, total_units + 1): # วนลูปหน่วยที่ 1 ถึง N
                c_unit = unit
                if district == 'นอกเขต':
                    c_unit = -1
                # print(sd,c_unit)
                # เช็คว่า 'เขต' ของหน่วยนี้มีใน df ไหม?
                if (sd, c_unit, 'เขต') not in existing_groups:
                    missing_whole_groups += 7
                    print(f"🚨 หายไปทั้งหน้า! [เขต] ตำบล: {sd} | หน่วยที่: {c_unit} (หายไป 7 รายชื่อ)")
                    
                # เช็คว่า 'บช' ของหน่วยนี้มีใน df ไหม?
                if (sd, c_unit, 'บช') not in existing_groups:
                    missing_whole_groups += 57
                    print(f"🚨 หายไปทั้งหน้า! [บช]  ตำบล: {sd} | หน่วยที่: {c_unit} (หายไป 57 รายชื่อ)")
    
    print("-" * 50)
    print(f"✅ จำนวนรายชื่อที่หายไปเพราะหายทั้งก้อน (ไม่ได้เข้า groupby): {missing_whole_groups}")

In [1308]:
check_missing_allrecord(df_result2)

🔍 ตามล่าหาข้อมูลที่หายไป 'ทั้งก้อน'...
--------------------------------------------------
✅ จำนวนรายชื่อที่หายไปเพราะหายทั้งก้อน (ไม่ได้เข้า groupby): 0


Mannual above file

In [1309]:
df_result2_again = pd.read_csv("master_result2.csv")

In [1310]:
check_from_df(df_result2_again)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

จำนวนที่ขาดรวมคิดเป็น: 0
✅ ข้อมูลสมบูรณ์แบบ! ทุกหน่วยมีรายชื่อครบถ้วน ไม่มีขาด ไม่มีเกิน ไม่มีซ้ำครับ


In [1311]:
df_result2_again.isnull().sum()

type              0
province          0
district          0
sub-district      0
unit_number       0
name              0
score           274
dtype: int64

In [1312]:
df_result2_again['score'] = df_result2_again['score'].fillna("0")

In [1313]:
df_result2_again.isnull().sum()

type            0
province        0
district        0
sub-district    0
unit_number     0
name            0
score           0
dtype: int64

In [1314]:
df_result2_again['score_clean'] = df_result2_again['score']

In [1315]:
def check_string_score(df):
    invalid_score_records = df[df['score_clean'].astype(str).str.contains(r'\D', regex=True, na=False)]

    # แสดงผลเรคคอร์ดที่พบปัญหา
    print(f"พบข้อมูลที่ไม่ใช่ตัวเลขจำนวน: {len(invalid_score_records)} แถว")
    print(invalid_score_records)

In [1316]:
check_string_score(df_result2_again)

พบข้อมูลที่ไม่ใช่ตัวเลขจำนวน: 2691 แถว
      type province district sub-district  unit_number            name score  \
45      บช    ลำปาง   นอกเขต     ชุดที่ 1           -1         ประชาชน  49.7   
256     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1    ไทยทรัพย์ทวี     -   
257     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1    เพื่อชาติไทย    A.   
258     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1            ใหม่    B.   
259     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1        มิติใหม่     -   
...    ...      ...      ...          ...          ...             ...   ...   
22713   บช    ลำปาง   แจ้ห่ม       แม่สุก            6   ประชาอาสาชาติ     -   
22714   บช    ลำปาง   แจ้ห่ม       แม่สุก            6           พร้อม     -   
22716   บช    ลำปาง   แจ้ห่ม       แม่สุก            6  ไทยพิทักษ์ธรรม     -   
22717   บช    ลำปาง   แจ้ห่ม       แม่สุก            6    ความหวังใหม่     -   
22718   บช    ลำปาง   แจ้ห่ม       แม่สุก            6       ไทยรวมไทย     -   



In [1317]:
df_result2_again.loc[df_result2_again['score_clean'].str.strip() == '-', 'score_clean'] = '0' 

In [1318]:
check_string_score(df_result2_again)

พบข้อมูลที่ไม่ใช่ตัวเลขจำนวน: 1093 แถว
      type province district sub-district  unit_number             name score  \
45      บช    ลำปาง   นอกเขต     ชุดที่ 1           -1          ประชาชน  49.7   
257     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1     เพื่อชาติไทย    A.   
258     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1             ใหม่    B.   
260     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1         รวมใจไทย    1.   
261     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1  รวมไทยสร้างชาติ    9.   
...    ...      ...      ...          ...          ...              ...   ...   
21131   บช    ลำปาง   แจ้ห่ม     ทุ่งผึ้ง            8           ไทยชนะ     /   
21240   บช    ลำปาง   แจ้ห่ม       แม่สุก           10       เสรีรวมไทย    .0   
21264   บช    ลำปาง   แจ้ห่ม       แม่สุก           10         ไทยพร้อม    .0   
21379   บช    ลำปาง   แจ้ห่ม       แม่สุก           12        ภูมิใจไทย   3.1   
21388   บช    ลำปาง   แจ้ห่ม       แม่สุก           12          ประชาช

In [1319]:
import pandas as pd
import re
from collections import defaultdict

def check_char2text(df):
    char_to_strings_map = defaultdict(list)

    # 2. ดึงเฉพาะข้อมูลที่ไม่ใช่ค่าว่าง (NaN) และลบค่าที่ซ้ำกันออกก่อน (เพื่อให้ทำงานเร็วขึ้น)
    unique_scores = df['score_clean'].dropna().astype(str).unique()

    # 3. วนลูปตรวจสอบทีละค่า
    for val in unique_scores:
        # re.findall(r'\D', val) จะดึง "ทุกอักขระที่ไม่ใช่ตัวเลข" ออกมาเป็นลิสต์
        # ใช้ set() ครอบอีกที เพื่อให้ได้อักขระที่ไม่ซ้ำกันใน 1 ข้อความ (เช่น " 10 " จะนับ space แค่ตัวเดียว)
        non_digits = set(re.findall(r'\D', val))

        for char in non_digits:
            char_to_strings_map[char].append(val)

    # 4. แสดงผลลัพธ์แบบสวยงาม
    print("🔍 สรุปตัวอักษรที่ไม่ใช่ตัวเลข และข้อความที่พบ:")
    print("-" * 60)
    for char, original_strings in char_to_strings_map.items():
        # จัด format ตัวอักษรให้ดูง่ายขึ้น (เช่น ถ้าเป็น space bar จะได้มองเห็น)
        display_char = f"[{char}]" if char == " " else f"'{char}'"

        print(f"อักขระ {display_char} : พบใน {len(original_strings)} รูปแบบข้อความ")

        # พิมพ์ตัวอย่างข้อความที่พบอักขระนี้ (โชว์สูงสุด 10 อันแรก จะได้ไม่รกจอเกินไป)
        print(f"   -> ตัวอย่าง: {original_strings[:10]}")
        print("-" * 60)
check_char2text(df_result2_again)

🔍 สรุปตัวอักษรที่ไม่ใช่ตัวเลข และข้อความที่พบ:
------------------------------------------------------------
อักขระ '.' : พบใน 60 รูปแบบข้อความ
   -> ตัวอย่าง: ['49.7', 'A.', 'B.', '1.', '9.', 'I.', '86.', '2.', '41.', '3.']
------------------------------------------------------------
อักขระ 'A' : พบใน 2 รูปแบบข้อความ
   -> ตัวอย่าง: ['A.', 'A']
------------------------------------------------------------
อักขระ 'B' : พบใน 2 รูปแบบข้อความ
   -> ตัวอย่าง: ['B.', 'B']
------------------------------------------------------------
อักขระ 'I' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['I.']
------------------------------------------------------------
อักขระ [ ] : พบใน 213 รูปแบบข้อความ
   -> ตัวอย่าง: ['10 (สม)', '6 (หก)', '109 (หนึ่งร้อยเก้า)', '19 (สิบเก้า)', '1 (หนึ่ง)', '200 (สองร้อย)', '70 (เจ็ดสิบ)', '5 ', '6 ', '3 ']
------------------------------------------------------------
อักขระ 'ม' : พบใน 20 รูปแบบข้อความ
   -> ตัวอย่าง: ['10 (สม)', '(สาม)', '(ยี่สิบสาม)', '( ส เม )', '( สุขุม )', '

In [1320]:
# df_result2_again['score_clean'] = df_result2_again['score_clean'].astype(str).str.replace('-', '0', regex=False)

# 2. ลบเฉพาะสิ่งที่ "มั่นใจว่าเป็นขยะแน่นอน" 
# (ภาษาไทย, เครื่องหมายวงเล็บ, และช่องว่าง \s)
df_result2_again['score_clean'] = df_result2_again['score_clean'].str.replace(r'[\u0E00-\u0E7F\(\)\s]', '', regex=True)

# 3. 🚨 สร้างตัวดักจับ (Flag): เช็คว่าผลลัพธ์ที่ได้ เป็นตัวเลขล้วนๆ (0-9) หรือไม่?
# ถ้าย้ายขยะปกติออกแล้ว ยังมีตัวประหลาด (เช่น 42+6 หรือ 12.5) มันจะไม่ match เงื่อนไขนี้
is_pure_number = df_result2_again['score_clean'].str.match(r'^\d+$')

# 4. แยกข้อมูลที่ "ต้องให้คน (Manual) ช่วยดู" ออกมา
# (คือพวกที่มีเครื่องหมายแปลกๆ หลงเหลืออยู่ หรือหาเลขไม่เจอเลย)
df_needs_manual = df_result2_again[~is_pure_number]

print(f"✅ ทำความสะอาดเป็นตัวเลขสมบูรณ์: {is_pure_number.sum()} แถว")
print(f"🚨 พบข้อมูลน่าสงสัย (ต้องเปิดไฟล์ภาพดู): {len(df_needs_manual)} แถว")

# ปริ้นดูว่ามีตัวไหนบ้างที่โค้ดฟ้องให้เราเช็ค
if len(df_needs_manual) > 0:
    print(df_needs_manual[['name', 'score', 'score_clean']])

✅ ทำความสะอาดเป็นตัวเลขสมบูรณ์: 21843 แถว
🚨 พบข้อมูลน่าสงสัย (ต้องเปิดไฟล์ภาพดู): 877 แถว
                  name score score_clean
45             ประชาชน  49.7        49.7
257       เพื่อชาติไทย    A.          A.
258               ใหม่    B.          B.
260           รวมใจไทย    1.          1.
261    รวมไทยสร้างชาติ    9.          9.
...                ...   ...         ...
21131           ไทยชนะ     /           /
21240       เสรีรวมไทย    .0          .0
21264         ไทยพร้อม    .0          .0
21379        ภูมิใจไทย   3.1         3.1
21388          ประชาชน  10.8        10.8

[877 rows x 3 columns]


In [1321]:
check_char2text(df_result2_again)

🔍 สรุปตัวอักษรที่ไม่ใช่ตัวเลข และข้อความที่พบ:
------------------------------------------------------------
อักขระ '.' : พบใน 51 รูปแบบข้อความ
   -> ตัวอย่าง: ['49.7', 'A.', 'B.', '1.', '9.', 'I.', '86.', '2.', '41.', '3.']
------------------------------------------------------------
อักขระ 'A' : พบใน 2 รูปแบบข้อความ
   -> ตัวอย่าง: ['A.', 'A']
------------------------------------------------------------
อักขระ 'B' : พบใน 2 รูปแบบข้อความ
   -> ตัวอย่าง: ['B.', 'B']
------------------------------------------------------------
อักขระ 'I' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['I.']
------------------------------------------------------------
อักขระ 'b' : พบใน 4 รูปแบบข้อความ
   -> ตัวอย่าง: ['b', '11b', 'bb', 'b.']
------------------------------------------------------------
อักขระ 'D' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['D']
------------------------------------------------------------
อักขระ 'O' : พบใน 2 รูปแบบข้อความ
   -> ตัวอย่าง: ['O', 'O.']
------------------------------------

In [1322]:
# แทนที่จุด '.' ด้วยค่าว่าง '' (ลบทิ้ง)
df_result2_again['score_clean'] = df_result2_again['score_clean'].astype(str).str.replace('.', '', regex=False)
check_char2text(df_result2_again)

🔍 สรุปตัวอักษรที่ไม่ใช่ตัวเลข และข้อความที่พบ:
------------------------------------------------------------
อักขระ 'A' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['A']
------------------------------------------------------------
อักขระ 'B' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['B']
------------------------------------------------------------
อักขระ 'I' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['I']
------------------------------------------------------------
อักขระ 'b' : พบใน 3 รูปแบบข้อความ
   -> ตัวอย่าง: ['b', '11b', 'bb']
------------------------------------------------------------
อักขระ 'D' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['D']
------------------------------------------------------------
อักขระ 'O' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['O']
------------------------------------------------------------
อักขระ '○' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['○']
------------------------------------------------------------
อักขระ '-' : พบใน 45 รูปแบบข้อความ
   -> ตัวอย่าง: ['1-', '0

In [1323]:
# 1. สร้างเงื่อนไขป้องกัน: เลือกเฉพาะแถวที่ไม่ได้มีแค่ '-' ตัวเดียวโดดๆ
mask_not_lone_dash = df_result2_again['score_clean'].astype(str).str.strip() != '-'

# 2. ตัดหาง: เจอ / หรือ - ตรงไหน ให้ลบทิ้งไปจนจบข้อความ
# r'[/-].*' หมายถึง "เจอ / หรือ - และอักขระอะไรก็ตามที่ตามหลังมัน (.*) ให้จับรวบแล้วลบทิ้งให้หมด"
df_result2_again.loc[mask_not_lone_dash, 'score_clean'] = (
    df_result2_again.loc[mask_not_lone_dash, 'score_clean']
    .astype(str)
    .str.replace(r'[/-].*', '', regex=True)
)

# 3. ลบช่องว่างหน้า-หลังที่อาจจะหลงเหลืออยู่ (เช่น "126 /-" พอโดนตัดอาจจะเหลือ "126 ")
df_result2_again['score_clean'] = df_result2_again['score_clean'].str.strip()

In [1324]:
check_string_score(df_result2_again)

พบข้อมูลที่ไม่ใช่ตัวเลขจำนวน: 121 แถว
      type province district sub-district  unit_number             name score  \
257     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1     เพื่อชาติไทย    A.   
258     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1             ใหม่    B.   
263     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1  ประชาธิปไตยใหม่    I.   
297     บช    ลำปาง   นอกเขต    ชุดที่ 13           -1         กล้าธรรม     A   
394     บช    ลำปาง   นอกเขต     ชุดที่ 3           -1         เศรษฐกิจ     b   
...    ...      ...      ...          ...          ...              ...   ...   
18693   บช    ลำปาง   แจ้ห่ม        ปงดอน            6        ภูมิใจไทย     A   
18697   บช    ลำปาง   แจ้ห่ม        ปงดอน            6      แผ่นดินธรรม     O   
18702   บช    ลำปาง   แจ้ห่ม        ปงดอน            6          ประชาชน     B   
19408   บช    ลำปาง   แจ้ห่ม     วิเชตนคร            7         เศรษฐกิจ     b   
21058   บช    ลำปาง   แจ้ห่ม     ทุ่งผึ้ง            7     ไทยทรัพย์ทวี

In [1325]:
check_char2text(df_result2_again)

🔍 สรุปตัวอักษรที่ไม่ใช่ตัวเลข และข้อความที่พบ:
------------------------------------------------------------
อักขระ 'A' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['A']
------------------------------------------------------------
อักขระ 'B' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['B']
------------------------------------------------------------
อักขระ 'I' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['I']
------------------------------------------------------------
อักขระ 'b' : พบใน 3 รูปแบบข้อความ
   -> ตัวอย่าง: ['b', '11b', 'bb']
------------------------------------------------------------
อักขระ 'D' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['D']
------------------------------------------------------------
อักขระ 'O' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['O']
------------------------------------------------------------
อักขระ '○' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['○']
------------------------------------------------------------
อักขระ 'C' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['C']
----

In [1326]:
df_result2_again.isnull().sum()

type            0
province        0
district        0
sub-district    0
unit_number     0
name            0
score           0
score_clean     0
dtype: int64

In [1327]:
df_result2_again.loc[(df_result2_again['score_clean'] == 'o') | (df_result2_again['score_clean'] == 'O'), 'score_clean'] = '0'

In [1328]:
check_string_score(df_result2_again)

พบข้อมูลที่ไม่ใช่ตัวเลขจำนวน: 56 แถว
      type province    district sub-district  unit_number              name  \
257     บช    ลำปาง      นอกเขต    ชุดที่ 13           -1      เพื่อชาติไทย   
258     บช    ลำปาง      นอกเขต    ชุดที่ 13           -1              ใหม่   
263     บช    ลำปาง      นอกเขต    ชุดที่ 13           -1   ประชาธิปไตยใหม่   
297     บช    ลำปาง      นอกเขต    ชุดที่ 13           -1          กล้าธรรม   
394     บช    ลำปาง      นอกเขต     ชุดที่ 3           -1          เศรษฐกิจ   
1908    บช    ลำปาง         งาว     บ้านร้อง           11          เศรษฐกิจ   
2103    บช    ลำปาง         งาว     บ้านร้อง            2           รักชาติ   
2104    บช    ลำปาง         งาว     บ้านร้อง            2          ไทยพร้อม   
2107    บช    ลำปาง         งาว     บ้านร้อง            2              กรีน   
2108    บช    ลำปาง         งาว     บ้านร้อง            2           ไทยธรรม   
2111    บช    ลำปาง         งาว     บ้านร้อง            2      พลังประชารัฐ   
2112    บช    ล

In [1329]:
check_char2text(df_result2_again)

🔍 สรุปตัวอักษรที่ไม่ใช่ตัวเลข และข้อความที่พบ:
------------------------------------------------------------
อักขระ 'A' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['A']
------------------------------------------------------------
อักขระ 'B' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['B']
------------------------------------------------------------
อักขระ 'I' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['I']
------------------------------------------------------------
อักขระ 'b' : พบใน 3 รูปแบบข้อความ
   -> ตัวอย่าง: ['b', '11b', 'bb']
------------------------------------------------------------
อักขระ 'D' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['D']
------------------------------------------------------------
อักขระ '○' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['○']
------------------------------------------------------------
อักขระ 'C' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['C']
------------------------------------------------------------
อักขระ '-' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['-']
----

In [1330]:
df_result2_again.loc[df_result2_again['score_clean'] == '-', :]

,type,province,district,sub-district,unit_number,name,score,score_clean
10319,บช,ลำปาง,วังเหนือ,วังทอง,7,เสรีรวมไทย,-.,-
10322,บช,ลำปาง,วังเหนือ,วังทอง,7,อนาคตไทย,-.,-
10326,บช,ลำปาง,วังเหนือ,วังทอง,7,สังคมประชาธิปไตย,-.,-
10327,บช,ลำปาง,วังเหนือ,วังทอง,7,ฟิวชัน,-.,-
10328,บช,ลำปาง,วังเหนือ,วังทอง,7,ไทรวมพลัง,-.,-
10330,บช,ลำปาง,วังเหนือ,วังทอง,7,ปวงชนไทย,-.,-
10335,บช,ลำปาง,วังเหนือ,วังทอง,7,ไทยก้าวหน้า,-.,-
10337,บช,ลำปาง,วังเหนือ,วังทอง,7,แรงงานสร้างชาติ,-.,-


In [1331]:
df_result2_again.loc[df_result2_again['score_clean'].str.strip() == '-', 'score_clean'] = '0' 
df_result2_again.loc[df_result2_again['score_clean'] == '-', :]

,type,province,district,sub-district,unit_number,name,score,score_clean


In [1332]:
df_result2_again.loc[df_result2_again['score_clean'] == "A", "score_clean"] = '4'

In [1333]:
df_result2_again.loc[df_result2_again['score_clean'] == "B", "score_clean"] = '3'
df_result2_again['score_clean'] = df_result2_again['score_clean'].astype(str).str.replace('b', '6', regex=False)

In [1334]:
check_char2text(df_result2_again)
check_string_score(df_result2_again)

🔍 สรุปตัวอักษรที่ไม่ใช่ตัวเลข และข้อความที่พบ:
------------------------------------------------------------
อักขระ 'I' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['I']
------------------------------------------------------------
อักขระ 'D' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['D']
------------------------------------------------------------
อักขระ '○' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['○']
------------------------------------------------------------
อักขระ 'C' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['C']
------------------------------------------------------------
อักขระ 'c' : พบใน 1 รูปแบบข้อความ
   -> ตัวอย่าง: ['c']
------------------------------------------------------------
พบข้อมูลที่ไม่ใช่ตัวเลขจำนวน: 13 แถว
      type province    district sub-district  unit_number             name  \
263     บช    ลำปาง      นอกเขต    ชุดที่ 13           -1  ประชาธิปไตยใหม่   
1908    บช    ลำปาง         งาว     บ้านร้อง           11         เศรษฐกิจ   
2103    บช    ลำปาง         งาว     

In [1335]:
df_result2_again.loc[df_result2_again['score_clean'] == "I", "score_clean"] = '1'
df_result2_again.loc[df_result2_again['score_clean'] == "D", "score_clean"] = '0'
df_result2_again.loc[df_result2_again['score_clean'] == "○", "score_clean"] = '0'
df_result2_again.loc[df_result2_again['score_clean'] == "C", "score_clean"] = '0'
df_result2_again.loc[df_result2_again['score_clean'] == "c", "score_clean"] = '0'

In [ ]:
df_result2_again.loc[df_result2_again['score_clean'] == '', 'score_clean'] = '0'

418       
419       
420       
421       
422       
        ..
20514     
20515     
20516     
20517     
21131     
Name: score_clean, Length: 246, dtype: str

In [1336]:
check_char2text(df_result2_again)
check_string_score(df_result2_again)

🔍 สรุปตัวอักษรที่ไม่ใช่ตัวเลข และข้อความที่พบ:
------------------------------------------------------------
พบข้อมูลที่ไม่ใช่ตัวเลขจำนวน: 0 แถว
Empty DataFrame
Columns: [type, province, district, sub-district, unit_number, name, score, score_clean]
Index: []


In [1338]:
mask_target = (df_result2_again['district'] == 'วังเหนือ') & (df_result2_again['sub-district'] == 'ทุ่งฮั้ว')

# 2. ดึงค่า unit_number ปัจจุบันออกมา และบังคับให้เป็นตัวเลข (int) เผื่อมันติดเป็น string อยู่
current_units = df_result2_again.loc[mask_target, 'unit_number'].astype(int)

# 3. ใช้คณิตศาสตร์สลับค่า: (จำนวนหน่วยสูงสุด + 1) - เลขเดิม -> ในที่นี้คือ 13 - เลขเดิม
reversed_units = 13 - current_units

# 4. เขียนค่าที่สลับแล้วกลับลงไปใน DataFrame
df_result2_again.loc[mask_target, 'unit_number'] = reversed_units

# ==========================================
# ตรวจสอบผลลัพธ์ว่าสลับถูกต้องหรือไม่
# ==========================================
print("✅ สลับหน่วย อำเภอวังเหนือ ตำบลทุ่งฮั้ว เรียบร้อยแล้ว!")
# สุ่มแสดงผลลัพธ์เพื่อตรวจสอบความถูกต้อง
print(df_result2_again[mask_target][['district', 'sub-district', 'unit_number', 'name', 'score_clean']].head(10))

✅ สลับหน่วย อำเภอวังเหนือ ตำบลทุ่งฮั้ว เรียบร้อยแล้ว!
      district sub-district  unit_number                          name  \
6519  วังเหนือ     ทุ่งฮั้ว           12  นางสาววิชุดา ว่องวัฒนวิโรจน์   
6520  วังเหนือ     ทุ่งฮั้ว           12          นางสาวสุวิภา กุศลจูง   
6521  วังเหนือ     ทุ่งฮั้ว           12              นายศรีพรหม หอมยก   
6522  วังเหนือ     ทุ่งฮั้ว           12           นายสมบูรณ์ รูปสะอาด   
6523  วังเหนือ     ทุ่งฮั้ว           12              นายดาชัย เอกปฐพี   
6524  วังเหนือ     ทุ่งฮั้ว           12           นายธนาธร โล่ห์สุนทร   
6525  วังเหนือ     ทุ่งฮั้ว           12    พันเอกสันทัด ภัทรกิตตินนท์   
6526  วังเหนือ     ทุ่งฮั้ว            3  นางสาววิชุดา ว่องวัฒนวิโรจน์   
6527  วังเหนือ     ทุ่งฮั้ว            3          นางสาวสุวิภา กุศลจูง   
6528  วังเหนือ     ทุ่งฮั้ว            3              นายศรีพรหม หอมยก   

     score_clean  
6519           6  
6520          59  
6521           5  
6522           3  
6523          85  
6524          44 

In [1339]:
check_from_df(df_result2_again)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

จำนวนที่ขาดรวมคิดเป็น: 0
✅ ข้อมูลสมบูรณ์แบบ! ทุกหน่วยมีรายชื่อครบถ้วน ไม่มีขาด ไม่มีเกิน ไม่มีซ้ำครับ


In [1345]:
df_result2_again['score_clean'] = df_result2_again['score_clean'].str.strip()

In [1346]:
df_result2_again['score_clean'] = df_result2_again['score_clean'].astype(int)

ValueError: invalid literal for int() with base 10: ''

In [1341]:
df_result2_again.to_csv("master_result3.csv", index=False)